The goal of this notebook is to perform a baseline RAG to detect which condition a patient has in the NTDS notes set. An LLM will be given a long note and will need to perform RAG on the specific note itself to help determine this. It should only retrieve from the specific note used.

In [1]:
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
import json
from langchain_google_genai import ChatGoogleGenerativeAI


/opt/miniconda3/envs/ntds-extractor/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.14) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
import re

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:


notes = pd.read_csv("data/synthetic_ntds_trauma_notes_gemini.csv", index_col = "encounter_id")

In [5]:
with open("./data/ntds_18_complications.json", "r") as f:
    complications_info = json.load(f)
complications_info

[{'id': 'aki',
  'label': 'Acute Kidney Injury',
  'short_definition': 'New kidney dysfunction during this hospitalization, such as a rise in serum creatinine or oliguria/anuria, not clearly present before admission.',
  'positive_note_clues': ['developed acute kidney injury with rising creatinine',
   'oliguria requiring nephrology consultation',
   'initiated dialysis for new renal failure']},
 {'id': 'aws',
  'label': 'Alcohol Withdrawal Syndrome',
  'short_definition': 'Clinical alcohol withdrawal that began after admission, with symptoms such as tremor, agitation, hallucinations, or withdrawal seizures.',
  'positive_note_clues': ['placed on alcohol withdrawal protocol with high CIWA scores',
   'developed agitation and tremors consistent with alcohol withdrawal',
   'treated with benzodiazepines for withdrawal symptoms']},
 {'id': 'ards',
  'label': 'Acute Respiratory Distress Syndrome',
  'short_definition': 'Acute hypoxemic respiratory failure with bilateral lung infiltrates no

In [6]:
mapping_dict = {complication['label']: complication['id'] for complication in complications_info}
mapping_dict

{'Acute Kidney Injury': 'aki',
 'Alcohol Withdrawal Syndrome': 'aws',
 'Acute Respiratory Distress Syndrome': 'ards',
 'Cardiac Arrest with CPR': 'cardiac_arrest_cpr',
 'Catheter-Associated Urinary Tract Infection': 'cauti',
 'Delirium': 'delirium',
 'Deep Venous Thrombosis / Thrombophlebitis': 'dvt',
 'Myocardial Infarction': 'mi',
 'Osteomyelitis': 'osteomyelitis',
 'Pressure Ulcer': 'pressure_ulcer',
 'Pulmonary Embolism': 'pe',
 'Severe Sepsis': 'severe_sepsis',
 'Stroke / Cerebrovascular Accident': 'stroke_cva',
 'Superficial Incisional Surgical Site Infection': 'superficial_ssi',
 'Unplanned Admission to ICU': 'unplanned_icu_admission',
 'Unplanned Intubation': 'unplanned_intubation',
 'Unplanned Visit to the Operating Room': 'unplanned_or_visit',
 'Ventilator-Associated Pneumonia': 'vap'}

In [7]:
sample_note = notes.loc[0, 'note_text']
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, 
                                          chunk_overlap = 200, 
                                          separators = ["\n\n", "\n", " ", ""])
chunked = splitter.split_text(sample_note)
chunked

['**ED TRAUMA H&P:**\nThis 79-year-old male was brought to the trauma bay via EMS after a significant fall from a height of approximately 15 feet while working on his roof. Per EMS report, he was found alert but confused at the scene by family and complained of diffuse body pain. On arrival, initial vital signs were heart rate 97 bpm, blood pressure 120/78 mmHg (MAP 92 mmHg), respiratory rate 22 breaths/min, and oxygen saturation 95% on a 4L nasal cannula, later titrated to 6L to maintain saturation >94%.',
 'Primary survey was completed rapidly per ATLS protocol. Airway was patent and protected. Breath sounds were clear bilaterally, though shallow, with no obvious respiratory distress. Cardiovascularly, peripheral pulses were palpable and strong, skin was warm and dry, capillary refill brisk. Neurologically, he presented with a GCS of 14 (E4V4M6), oriented to person but confused to place and time, without obvious focal deficits upon initial assessment. Gross deformities were noted to 

In [8]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
import os

# Initialize ChromaDB with persistence
PERSIST_DIRECTORY = "./ntds_embeddings"

embeddings = OllamaEmbeddings(model="llama3.1:8b")

vectorstore = Chroma(
    collection_name="ntds_notes",
    embedding_function=embeddings,
    persist_directory=PERSIST_DIRECTORY
)

# Initialize text splitter with same parameters as before
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200, 
    separators=["\n\n", "\n", " ", ""]
)

# Process all notes
all_chunks = []
all_metadatas = []

print(f"Processing {len(notes)} notes...")
for idx, (encounter_id, row) in enumerate(notes.iterrows()):
    note_text = row['note_text']
    
    # Chunk the note
    chunks = splitter.split_text(note_text)
    
    # Create metadata for each chunk
    for chunk_idx, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        all_metadatas.append({
            'encounter_id': str(encounter_id),
            'chunk_index': chunk_idx,
            'total_chunks': len(chunks)
        })
    
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1}/{len(notes)} notes...")

print(f"\nAdding {len(all_chunks)} chunks to ChromaDB...")
vectorstore.add_texts(texts=all_chunks, metadatas=all_metadatas)

print(f"Successfully created ChromaDB with {len(all_chunks)} chunks")
print(f"Persisted to: {PERSIST_DIRECTORY}")

Processing 50 notes...
Processed 10/50 notes...
Processed 20/50 notes...
Processed 30/50 notes...
Processed 40/50 notes...
Processed 50/50 notes...

Adding 141 chunks to ChromaDB...
Successfully created ChromaDB with 141 chunks
Persisted to: ./ntds_embeddings


## RAG with Metadata Filtering

Now we'll implement single-note RAG using ChromaDB's metadata filtering. This ensures we only retrieve chunks from the specific patient note being analyzed.

In [9]:
# Reload the existing vectorstore (no need to recreate)
vectorstore = Chroma(
    collection_name="ntds_notes",
    embedding_function=embeddings,
    persist_directory=PERSIST_DIRECTORY
)

print(f"Loaded vectorstore with collection: ntds_notes")
print(f"Total documents in collection: {vectorstore._collection.count()}")

Loaded vectorstore with collection: ntds_notes
Total documents in collection: 282


In [10]:
# Test metadata filtering - retrieve only from encounter_id 0
test_encounter_id = "0"
test_query = "kidney injury or renal failure"

print(f"Testing retrieval for encounter {test_encounter_id}")
print(f"Query: '{test_query}'")
print("="*60)

match = vectorstore.similarity_search(
    test_query,
    k=5,
    filter={"encounter_id": test_encounter_id}
)


Testing retrieval for encounter 0
Query: 'kidney injury or renal failure'


In [11]:
def build_complication_query(complication: dict) -> str:
    """Build focused retrieval query for a single complication. Includes the name, definition,
    and clues
    
    Args:
        complication: Dict with keys 'label', 'short_definition', 'positive_note_clues'
    
    Returns:
        str: Combined query string for retrieval
    """
    query_parts = [
        complication['label'],
        complication['short_definition'],
        ' '.join(complication['positive_note_clues'])
    ]
    return ' '.join(query_parts)


In [12]:
# Helper function to format the prompt with complication info
output_format_template = """At the end of your message, Output the following as JSON:
```json
{{{{
    "{condition_name}": "<Yes or No>"
}}}}
```
"""

def format_complication_prompt(complication: dict, retrieved_chunks: str):
    """Format the per-complication prompt with complication dict and retrieved chunks.

    Args:
        complication: Dict with keys 'label', 'short_definition', 'positive_note_clues'
        retrieved_chunks: String containing retrieved and formatted chunks

    Returns:
        list: Formatted messages ready for LLM
    """
    
    output_instructions = output_format_template.format(
        condition_name=complication['label']
    )

    # Format positive clues
    clues_text = "\n".join([
        f"  {i+1}. {clue}"
        for i, clue in enumerate(complication['positive_note_clues'])
    ])

    # Build system message using concatenation instead of f-string to avoid conflicts
    system_message = """You are an expert registrar who is highly experienced at meeting the
National Trauma Data Standard (NTDS). You are analyzing a patient's medical note from
UCSD Health, a level 1 trauma center. Your task is to determine whether this patient
has a SPECIFIC complication. Go step by step and state your train of 

COMPLICATION TO ASSESS:
- Name: """ + complication['label'] + """
- Definition: """ + complication['short_definition'] + """
- Positive Indicators (what to look for):
""" + clues_text + """

""" + output_instructions

    per_complication_template = ChatPromptTemplate([
        ("system", system_message),
        ("human", "{retrieved_chunks}"),
        ("system", "Based on the retrieved chunks above, provide your assessment.")
    ])

    return per_complication_template.format_messages(
        retrieved_chunks=retrieved_chunks
    )


In [13]:
def retrieve_relevant_chunks(vectorstore, encounter_id, query, k=5):
    results = vectorstore.similarity_search(
        query,
        k=k,
        filter={"encounter_id": str(encounter_id)}
    )
    return [doc.page_content for doc in results]


## Per-Complication RAG Demonstration

Now we'll demonstrate the per-complication approach with two examples:
1. **Acute Kidney Injury (AKI)** - A biomarker-based condition
2. **Unplanned ICU Admission** - A contextual/narrative-based condition

Each complication will get its own focused query and individual LLM call.

In [14]:
def assess_single_complication(
    complication: dict, encounter_id: str|int, vectorstore: Chroma, llm, k=2, verbose=False):
    """Assess whether a patient has a specific complication using RAG.
    
    Important Args:
        complication: Dict with keys 'id', 'label', 'short_definition', 'positive_note_clues'
        k: Number of chunks to retrieve (default: 2)
        verbose: Whether to print detailed progress (default: False)
    
    Returns:
        str: LLM response (JSON string with assessment)
    """
    if verbose:
        print(f"\n{'='*70}")
        print(f"Assessing: {complication['label']} (ID: {complication['id']})")
        print(f"{'='*70}")
    
    # 1. Build focused query
    query = build_complication_query(complication)
    if verbose:
        print(f"\n1. Built query ")
    
    # 2. Retrieve relevant chunks
    chunks = retrieve_relevant_chunks(vectorstore, encounter_id, query, k=k)
    if verbose:
        print(f"\n2. Retrieved {len(chunks)} chunks for encounter {encounter_id}")
        print("\n   Preview of first chunk:")
        print(f"   {chunks[0][:200]}..." if chunks else "   No chunks retrieved")
    
    # 3. Format context
    retrieved_context = "\n\n---\n\n".join([
        f"Relevant Section {i+1}:\n{chunk}"
        for i, chunk in enumerate(chunks)
    ])
    if verbose:
        print(f"\n3. Formatted context (length: {len(retrieved_context)} chars)")
    
    # 4. Format messages using helper function
    messages = format_complication_prompt(complication, retrieved_context)
    if verbose:
        print(f"\n4. Prepared LLM prompt with {len(messages)} messages")

    # 5. Call LLM
    if verbose:
        print("\n5. Calling LLM...")
    response = llm.invoke(messages)
    
    # 6. Return result
    if verbose:
        print(f"\n6. LLM Response:\n{response.content}")
    
    return response.content



llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
# encounter_id = "0"

# aki_info = complications_info[0]  

# # Use the new function
# label = assess_single_complication(
#     complication=aki_info,
#     encounter_id=encounter_id,
#     vectorstore=vectorstore,
#     llm=llm,
#     k=5,
#     verbose=True
# )

# print("Final Result:")
# print(label)

# json_regex = r"\`\`\`json\n(.*?)\s*\`\`\`"
# match = re.search(json_regex, label, flags = re.DOTALL)
# out = json.loads(match.group(1))['Acute Kidney Injury']

In [15]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
NUM_COMPLICATIONS_NTDS_18 = 18
ALL_COMPLICATIONS_RANGE = (0, NUM_COMPLICATIONS_NTDS_18)
def get_all_complications_for_encounter(encounter_id, k = 2, complications_range = ALL_COMPLICATIONS_RANGE):
    outputs_dict = dict()
    for i in range(*complications_range):
        complication = complications_info[i]
        complication_label = complication['label']
        label = assess_single_complication(
        complication=complication,
        encounter_id=encounter_id,
        vectorstore=vectorstore,
        llm=llm,
        k=k)

        json_regex = r"\`\`\`json\n(.*?)\s*\`\`\`"
        match = re.search(json_regex, label, flags = re.DOTALL)
        if match is not None:
            out = json.loads(match.group(1))[complication_label]
            if out.casefold() == "no":
                outputs_dict[complication_label] = False
            elif out.casefold() == "yes":
                outputs_dict[complication_label] = True         
            else:
                raise RuntimeError(f"get_all_outputs_for_encounter(): Complication Label should be yes or no. Instead it is {out}")
        else:
            raise RuntimeError(f"get_all_outputs_for_encounter(): No match found for i = {i}")
        print(outputs_dict)
    return outputs_dict

In [16]:
ENCOUNTER_ID = 1
outputs_dict = get_all_complications_for_encounter(encounter_id = ENCOUNTER_ID, k = 2, complications_range= (3, 10))
outputs_dict

{'Cardiac Arrest with CPR': False}
{'Cardiac Arrest with CPR': False, 'Catheter-Associated Urinary Tract Infection': False}
{'Cardiac Arrest with CPR': False, 'Catheter-Associated Urinary Tract Infection': False, 'Delirium': False}
{'Cardiac Arrest with CPR': False, 'Catheter-Associated Urinary Tract Infection': False, 'Delirium': False, 'Deep Venous Thrombosis / Thrombophlebitis': False}
{'Cardiac Arrest with CPR': False, 'Catheter-Associated Urinary Tract Infection': False, 'Delirium': False, 'Deep Venous Thrombosis / Thrombophlebitis': False, 'Myocardial Infarction': False}
{'Cardiac Arrest with CPR': False, 'Catheter-Associated Urinary Tract Infection': False, 'Delirium': False, 'Deep Venous Thrombosis / Thrombophlebitis': False, 'Myocardial Infarction': False, 'Osteomyelitis': False}
{'Cardiac Arrest with CPR': False, 'Catheter-Associated Urinary Tract Infection': False, 'Delirium': False, 'Deep Venous Thrombosis / Thrombophlebitis': False, 'Myocardial Infarction': False, 'Osteomy

{'Cardiac Arrest with CPR': False,
 'Catheter-Associated Urinary Tract Infection': False,
 'Delirium': False,
 'Deep Venous Thrombosis / Thrombophlebitis': False,
 'Myocardial Infarction': False,
 'Osteomyelitis': False,
 'Pressure Ulcer': True}

In [23]:
def compare_outputs_and_score(llms_outputs: dict, true_outputs: dict, mapping_dict: dict):
    # Convert LLM outputs from labels to ids. Returns a tuple of (num_correct, num_incorrect)
    num_correct = 0
    num_incorrect = 0
    llm_output_with_ids = {mapping_dict[key]: val for key, val in llms_outputs.items()}
    
    for key in llm_output_with_ids:

        if true_outputs[key] == llm_output_with_ids[key]:
            num_correct += 1
        else:
            num_incorrect += 1
    return num_correct, num_incorrect

In [24]:
FIRST_OUTPUT_COL = 8 # First column in dataframe with outputs (aki
DF_RANGE = (FIRST_OUTPUT_COL + ALL_COMPLICATIONS_RANGE[0], FIRST_OUTPUT_COL + ALL_COMPLICATIONS_RANGE[1])
temp_dict = dict(notes.iloc[ENCOUNTER_ID, DF_RANGE[0]: DF_RANGE[1]])
true_values = {key: bool(val) for key, val in temp_dict.items()}
true_values

{'aki': False,
 'aws': False,
 'ards': False,
 'cardiac_arrest_cpr': False,
 'cauti': False,
 'delirium': False,
 'dvt': False,
 'mi': True,
 'osteomyelitis': False,
 'pressure_ulcer': True,
 'pe': False,
 'severe_sepsis': False,
 'stroke_cva': False,
 'superficial_ssi': False,
 'unplanned_icu_admission': False,
 'unplanned_intubation': False,
 'unplanned_or_visit': False,
 'vap': False}

In [25]:
outputs_dict

{'Cardiac Arrest with CPR': False,
 'Catheter-Associated Urinary Tract Infection': False,
 'Delirium': False,
 'Deep Venous Thrombosis / Thrombophlebitis': False,
 'Myocardial Infarction': False,
 'Osteomyelitis': False,
 'Pressure Ulcer': True}

In [26]:
true_values

{'aki': False,
 'aws': False,
 'ards': False,
 'cardiac_arrest_cpr': False,
 'cauti': False,
 'delirium': False,
 'dvt': False,
 'mi': True,
 'osteomyelitis': False,
 'pressure_ulcer': True,
 'pe': False,
 'severe_sepsis': False,
 'stroke_cva': False,
 'superficial_ssi': False,
 'unplanned_icu_admission': False,
 'unplanned_intubation': False,
 'unplanned_or_visit': False,
 'vap': False}

In [27]:
correct_values, incorrect_values = compare_outputs_and_score(outputs_dict, true_values, mapping_dict)

In [29]:
correct_values, incorrect_values

(6, 1)

The model correctly labels all conditions as False.